In [3]:
!nvidia-smi
!pip install -q -U ultralytics
import ultralytics
print(ultralytics.__version__)

Fri Oct 23 05:53:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os, yaml
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

FYP = "/content/drive/MyDrive/FYP"
DATA = "/content/data_v3"
if not os.path.exists(f"{DATA}/data.yaml"):
    !unzip -q -o "{FYP}/merged_subset_v3.zip" -d {DATA}

cfg = yaml.safe_load(open(f"{DATA}/data.yaml"))
cfg["path"] = DATA
yaml.safe_dump(cfg, open(f"{DATA}/data.yaml", "w"), sort_keys=False)
for s in ["train", "valid", "test"]:
    print(s, len(os.listdir(f"{DATA}/images/{s}")), "images")
print(cfg["nc"], "classes")

Mounted at /content/drive
train 12000 images
valid 1000 images
test 700 images
22 classes


In [ ]:
from ultralytics import YOLO

RUN = "colab-run6-26n"
PROJECT = f"{FYP}/runs"
last = f"{PROJECT}/{RUN}/weights/last.pt"

if os.path.exists(last):
    print("resuming from", last)
    YOLO(last).train(resume=True)
else:
    YOLO("yolo26n.pt").train(data=f"{DATA}/data.yaml", epochs=40, imgsz=640, batch=32,
                             workers=2, project=PROJECT, name=RUN, exist_ok=True, patience=15)

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_v3/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=colab-run6-26n, nbs=64, nms=None, opset=None, optimize=False, op

In [ ]:
import pandas as pd

m = YOLO(f"{PROJECT}/{RUN}/weights/best.pt")
r = m.val(data=f"{DATA}/data.yaml", split="test", imgsz=640, batch=32, plots=True)

rows = []
for k, c in enumerate(r.box.ap_class_index):
    p, rec, ap50, ap = r.box.class_result(k)
    rows.append({"class": m.names[int(c)], "precision": p, "recall": rec, "mAP50": ap50, "mAP50-95": ap})
rows.append({"class": "ALL", "precision": r.box.mp, "recall": r.box.mr, "mAP50": r.box.map50, "mAP50-95": r.box.map})
df = pd.DataFrame(rows).round(3)
df.to_csv(f"{PROJECT}/{RUN}/test_per_class.csv", index=False)
print(df.to_string(index=False))
print("ms per image:", {k: round(v, 1) for k, v in r.speed.items()})